In [ ]:
import os, sys, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(sys.executable)
# sys.executable = "./scripts/tf_gpu_env.fish
print("LD_LIBRARY_PATH contains nvidia:", "site-packages/nvidia" in os.environ.get("LD_LIBRARY_PATH",""))

USE_GPU = True
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "0"
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
if not gpus and USE_GPU:
    raise RuntimeError("No GPU detected by TensorFlow. Refusing CPU fallback.")
else:
    tf.config.set_visible_devices(gpus[0], "GPU")
    tf.config.experimental.set_memory_growth(gpus[0], True)


/home/kilo/Work/Cours - UQO/concept-statistique/concept-stat-projet-1/.venv-stat/bin/python3.11
LD_LIBRARY_PATH contains nvidia: True
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:

# Notebook  : Réseaux de Neurones Convolutionnels (CNN) pour la Classification d’Images



# Charger le dataset CIFAR-10
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Normalisation des images
x_train, x_test = x_train / 255.0, x_test / 255.0

# Définition du modèle CNN
model = keras.Sequential([
    layers.Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(32, 32, 3)),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Conv2D(128, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

# Compilation du modèle
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Entraînement rapide pour test (les étudiants complèteront)
model.fit(x_train, y_train, epochs=1, batch_size=32, validation_data=(x_test, y_test))

# Évaluation du modèle
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f'Précision sur l’ensemble de test : {test_acc:.4f}')



# Question pour expérimenter l'Apprentissage par Transfert




GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
A local file was found, but it seems to be incomplete or outdated because the auto file hash does not match the original value of 6d958be074577803d12ecdefd02955f39262c83c16fe9348329d7fe0b5c001ce so we will re-download the data.
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 52s 0us/step


/home/kilo/Work/Cours - UQO/concept-statistique/concept-stat-projet-1/.venv-stat/lib/python3.11/site-packages/keras/src/datasets/cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")
/home/kilo/Work/Cours - UQO/concept-statistique/concept-stat-projet-1/.venv-stat/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
W0000 00:00:1775607282.466117 1816640 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minu

  24/1563 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.0975 - loss: 2.2953  

I0000 00:00:1775607299.396537 1819544 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1541/1563 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3505 - loss: 1.7495

I0000 00:00:1775607308.005791 1819542 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1930__.39


1563/1563 ━━━━━━━━━━━━━━━━━━━━ 29s 11ms/step - accuracy: 0.4480 - loss: 1.5126 - val_accuracy: 0.5698 - val_loss: 1.2057
313/313 - 2s - 6ms/step - accuracy: 0.5698 - loss: 1.2057
Précision sur l’ensemble de test : 0.5698


NameError: name 'applications' is not defined

In [ ]:
# Charger le modèle pré-entraîné  VGG16
base_model = applications.VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))
base_model.trainable = False  # Geler les poids du modèle pré-entraîné

# Ajouter une couche de classification personnalisée
transfer_model = keras.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

# Compilation et entraînement du modèle avec transfert d’apprentissage
transfer_model.compile(optimizer='adam',
                        loss='sparse_categorical_crossentropy',
                        metrics=['accuracy'])

# Entraînement rapide pour test
transfer_model.fit(x_train, y_train, epochs=1, batch_size=32, validation_data=(x_test, y_test))

# Comparer la performance avec et sans transfert d’apprentissage
test_loss_transfer, test_acc_transfer = transfer_model.evaluate(x_test, y_test, verbose=2)
print(f'Précision avec apprentissage par transfert : {test_acc_transfer:.4f}')